## load Python libs

In [94]:
# -*- coding: utf-8 -*-
"""
Created on Tue Apr 28 11:29:58 2026

@author: MountainViewLabUser
"""
import os
import sys
from tqdm import tqdm
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
#import ips_clm_tile_control
import time
from datetime import datetime, timezone
import pyvisa as visa
import yaml
import pickle
import random

#sys.path.append(r'C:\Users\MountainViewLabUser\gitrepos\instrument-drivers\src')
sys.path.append(r'C:\Users\MountainViewLabUser\gitrepos\instrument-drivers\src\instrument_drivers\devices')
#DRIVER_FOLDER = r"C:\Users\MountainViewLabUser\xingpan\scripts\instr"
#if DRIVER_FOLDER not in sys.path:
#    sys.path.append(DRIVER_FOLDER)
from keysight_N7745c_opm import KeysightN7745C as OPM

#DRIVER_FOLDER = r"C:\Users\MountainViewLabUser\gitrepos\instrument-drivers"
#if DRIVER_FOLDER not in sys.path:
#    sys.path.append(DRIVER_FOLDER)
#DRIVER_FOLDER = r'C:\Users\MountainViewLabUser\gitrepos\instrument-drivers\src\instrument_drivers'
#if DRIVER_FOLDER not in sys.path:
#    sys.path.append(DRIVER_FOLDER)
#from Keysight_N776xC import KeysightN776xC as VOA

#from Thorlabs_PM101 import ThorPM101

venv_scripts = r'C:\Users\MountainViewLabUser\gitrepos\cmis-host-controller\.venv\Scripts'
if os.path.exists(venv_scripts):
    os.add_dll_directory(venv_scripts)
    
import concurrent.futures
MARLOW_API_PATH=r'C:\Users\MountainViewLabUser\gitrepos\cmis-host-controller\src\cmis_host\platform\clm\scripts'
if MARLOW_API_PATH not in sys.path:
    sys.path.append(MARLOW_API_PATH)
import marlow_clm_control_api as mca
from  pptx_maker import PresentationManager

#from pandasgui import show
sns.set_palette("bright")

def export_to_yaml_file(dataDir,dict_fru_revised,sn='dummy',operator='dummy'):
    '''
    '''
    os.chdir(dataDir)
    t0 = datetime.now()
    timeStamp = t0.strftime("%Y-%m-%d-%H-%M-%S")

    for k,dictToDump in dict_fru_revised.items():
        yaml_string = yaml.dump(dictToDump)
        yamlFileName='_'.join([sn,f'tile{k}',dictToDump['serial_number'],operator,timeStamp,'.yaml'])
        with open(yamlFileName, 'w') as file:
            yaml.dump(dictToDump, file, default_flow_style=False)
        
    return 0

rm=visa.ResourceManager()

In [95]:
venv_scripts = r'C:\Users\MountainViewLabUser\gitrepos\cmis-host-controller\.venv\Scripts'
if os.path.exists(venv_scripts):
    os.add_dll_directory(venv_scripts)
    
import concurrent.futures
#MARLOW_API_PATH=r'C:\Users\XingPan\gitrepos\congo-clm-fw\cmis-host-controller\src\cmis_host\platform\clm\scripts'
MARLOW_API_PATH=r'C:\Users\MountainViewLabUser\gitrepos\cmis-host-controller\src\cmis_host\platform\clm\scripts'
if MARLOW_API_PATH not in sys.path:
    sys.path.append(MARLOW_API_PATH)
import marlow_clm_control_api as mca

In [96]:
testBoardType='EVK' #'KLM', 'CNT', 'EVK','CLM-M'
AARDVARK_SN_P0=2239820367 # 2239867798 #2239836767 # 
AARDVARK_SN_Io=2239813795 # 2239880394 #2239833479 # 
tb=mca.get_testboard(AARDVARK_SN_P0 = AARDVARK_SN_P0,AARDVARK_SN_Io = AARDVARK_SN_Io,AARDVARK_SN_P1 =2239789566,testBoardType=testBoardType)

CommunicationError: Could not open Aardvarks

## Please provide information

In [ ]:
sn='nnnn' # sn of the clm package
operator='xing' # name of operator
#############
'''
information below need not to be changed. if not sure, please check with TPM or responsible engineer
'''
dir_fw=r'C:\Users\MountainViewLabUser\gitrepos\clm_bin_files\bin_files'
#filename_fw='clm_rel_marlow_tile_v1.8.bin'
filename_fw='clm_rel_marlow_tile_v1.14.bin'
#filename_fw='clm_rel_marlow_tile_v1.11.bin'
#filename_fw='clm_rel_marlow_tile_v1.10-TEST-8_lsb_amplitude.bin'
#filename_fw='clm_rel_marlow_tile_v1.10.bin'
#filename='clm_rel_marlow_tile_v1.10-TEST-8_dac_amplitude.bin'
#filename='clm_rel_marlow_tile_v1.10-TEST_25_dac_amplitude.bin'

power_tgt=12
power_target=power_tgt
I_VOA_0=15
bRevise_init_Tpic=True

kps=[0.2,0.2,0.2]
ditherAmp=3

lambda0_voa=1310
attn0_dB=20

## optical power on
* before executing next cell, make sure clm output fibers are connected properly

In [ ]:
tile=tb.module.packages[0].tiles[3]
mca.initialize_tile(tb,tile=tile)
mca.set_clm_output_targets(tile,freq_offset_targets=[0]*16,power_targets=[power_target]*16)
mca.set_clm_control_config(tile,dither_period_s=2,kp_tpic=kps[0],kp_ld=kps[1],kp_voa=kps[2],amp_t_mux=ditherAmp,ALPHA_I_LD=0.5)
time.sleep(0.2)
mca.optical_power_on(tile)

## control channel enable/disable

In [97]:
chMask=[1,1,1,1,1,1,1,1]+[1,1,1,1,1,1,1,1]
#chMask=[0]*16
mca.clm_lane_enable(tile,chMask)

0

In [ ]:
#mca.shut_down_test_board_nicely(tb)
